## 7. Granger Causality Screening

Granger tests are retained as predictive screens only. A significant result means past values of a predictor contain incremental predictive information for CPI inflation conditional on past CPI inflation. Granger causality is not structural or economic causality, especially for policy variables such as the RBA cash rate that also respond to inflation.

In [9]:
def compute_granger_screens(
    target: str,
    predictor_columns: list[str],
    target_stationary_values: pd.Series,
    best_predictive: pd.DataFrame,
    max_lag_limit: int = MAX_LAG,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, int]]:
    best_lags = dict(zip(best_predictive['variable'], best_predictive['lag_quarters']))

    def granger_screen(variable: str) -> dict[str, object]:
        x = stationary_series(variable).rename(variable)
        data = pd.concat([target_stationary_values.rename(target), x], axis=1).dropna()
        max_lag = min(max_lag_limit, max(1, len(data) // 10))
        selected_lag = int(best_lags.get(variable, 1))
        selected_lag = min(max(selected_lag, 1), max_lag)
        base_record = {
            'target': target,
            'variable': variable,
            'selected_lag': selected_lag,
            'granger_p_value': np.nan,
            'min_granger_p_value': np.nan,
            'tested_lags': '',
            'n_obs': len(data),
            'note': 'insufficient stationary observations',
            'p_values_by_lag': {},
        }
        if len(data) < 24 or data[variable].nunique() < 3:
            return base_record
        try:
            result = grangercausalitytests(data[[target, variable]], maxlag=max_lag, verbose=False)
            p_values = {lag: float(result[lag][0]['ssr_ftest'][1]) for lag in range(1, max_lag + 1)}
            base_record.update({
                'granger_p_value': p_values.get(selected_lag, np.nan),
                'min_granger_p_value': min(p_values.values()) if p_values else np.nan,
                'tested_lags': ', '.join(f'{lag}:{p_value:.3f}' for lag, p_value in p_values.items()),
                'note': 'screening signal only; not structural causality',
                'p_values_by_lag': p_values,
            })
            return base_record
        except Exception as exc:
            base_record['note'] = f'granger failed: {type(exc).__name__}'
            return base_record

    records = [granger_screen(column) for column in predictor_columns]
    granger_frame = pd.DataFrame([{key: value for key, value in record.items() if key != 'p_values_by_lag'} for record in records])
    lag_results = pd.DataFrame([
        {
            'target': record['target'],
            'variable': record['variable'],
            'lag_quarters': lag,
            'granger_p_value': p_value,
            'n_obs': record['n_obs'],
        }
        for record in records
        for lag, p_value in record.get('p_values_by_lag', {}).items()
    ])
    return granger_frame, lag_results, {variable: int(lag) for variable, lag in best_lags.items()}


def display_granger_screens(granger_frame: pd.DataFrame, lag_results: pd.DataFrame) -> None:
    display(granger_frame.sort_values('granger_p_value', na_position='last').round(4))
    display(lag_results.sort_values(['variable', 'lag_quarters']).round(4))


granger, granger_lag_results, best_lag_lookup = compute_granger_screens(
    TARGET,
    CORRELATION_BASE_COLS,
    target_stationary,
    best_predictive_ccf,
)
display_granger_screens(granger, granger_lag_results)

,target,variable,selected_lag,granger_p_value,min_granger_p_value,tested_lags,n_obs,note
0,cpi_yoy,cpi_qoq,2,0.0000,0.0000,"1:0.000, 2:0.000, 3:0.000, 4:0.000",124,screening signal only; not structural causality
22,cpi_yoy,inflation_expectations_business,1,0.0000,0.0000,"1:0.000, 2:0.000, 3:0.000, 4:0.000",124,screening signal only; not structural causality
6,cpi_yoy,cash_rate,2,0.0001,0.0000,"1:0.000, 2:0.000, 3:0.001, 4:0.002",123,screening signal only; not structural causality
7,cpi_yoy,cash_rate_change,2,0.0001,0.0000,"1:0.000, 2:0.000, 3:0.001, 4:0.002",123,screening signal only; not structural causality
15,cpi_yoy,wti_growth,3,0.0001,0.0000,"1:0.002, 2:0.004, 3:0.000, 4:0.000",101,screening signal only; not structural causality
14,cpi_yoy,wti_price,3,0.0003,0.0000,"1:0.003, 2:0.006, 3:0.000, 4:0.000",101,screening signal only; not structural causality
17,cpi_yoy,brent_growth,3,0.0012,0.0002,"1:0.000, 2:0.007, 3:0.001, 4:0.000",73,screening signal only; not structural causality
21,cpi_yoy,household_spending_growth,3,0.0016,0.0016,"1:0.107, 2:0.103, 3:0.002, 4:0.013",53,screening signal only; not structural causality
20,cpi_yoy,household_spending,3,0.0037,0.0037,"1:0.132, 2:0.145, 3:0.004, 4:0.018",53,screening signal only; not structural causality
11,cpi_yoy,ppi_growth,2,0.0066,0.0066,"1:0.050, 2:0.007, 3:0.087, 4:0.142",109,screening signal only; not structural causality


,target,variable,lag_quarters,granger_p_value,n_obs
72,cpi_yoy,aud_usd,1,0.1987,63
73,cpi_yoy,aud_usd,2,0.4722,63
74,cpi_yoy,aud_usd,3,0.5591,63
75,cpi_yoy,aud_usd,4,0.8359,63
76,cpi_yoy,aud_usd_change,1,0.1972,63
...,...,...,...,...,...
63,cpi_yoy,wti_growth,4,0.0000,101
56,cpi_yoy,wti_price,1,0.0027,101
57,cpi_yoy,wti_price,2,0.0059,101
58,cpi_yoy,wti_price,3,0.0003,101
